# Classification Metrics - Confusion Matrix, ROC/PR, and Business Costs

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/07_classification_metrics_thresholding_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Compute and interpret precision, recall, F1, ROC-AUC, PR-AUC
2. Select thresholds based on business cost tradeoffs
3. Handle class imbalance at the evaluation level (metrics first)
4. Produce a metrics dashboard table for model comparison

*(Calibration — whether the probabilities you are thresholding are actually trustworthy — returns in **NB16 (Decision Thresholds & Calibration)**, once you have seen classifiers that can genuinely be miscalibrated. Logistic regression is natively well-calibrated by its loss function, so deferring the topic to NB16 keeps the pedagogy aligned with where the problem actually bites.)*

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: When 95% Accuracy Isn't Good Enough

Your logistic regression for the **State Health Department** achieves 95% accuracy on the breast cancer data. The chief medical officer calls an urgent meeting: *"How many malignant cases did we miss?"* You check — and discover that 3 malignant tumors were classified as benign.

In medicine, a missed cancer (false negative) is catastrophic: delayed treatment, metastasis, potentially death. An unnecessary biopsy (false positive) causes stress and cost, but the patient survives. Accuracy alone cannot capture this asymmetry. You need metrics that distinguish between types of errors and a threshold that reflects the real-world cost of each mistake.

> **Today's focus:** Moving beyond accuracy to precision, recall, F1, and ROC analysis, and learning to set classification thresholds based on the cost of errors.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell imports every metric the Health Department will need to evaluate MedScreen: `confusion_matrix` for the raw error breakdown, `precision_score` and `recall_score` for the question "how many cancers did we catch?", `roc_auc_score` and `roc_curve` for threshold-independent model quality, and `precision_recall_curve` with `average_precision_score` for evaluation under class imbalance. The display helpers — `ConfusionMatrixDisplay`, `RocCurveDisplay`, `PrecisionRecallDisplay` — produce publication-quality plots with a single call, ready for a board presentation. The confirmation **"Setup complete!"** means all libraries loaded without conflict.

**Why this matters:** Having every metric function imported up front keeps the rest of the notebook focused on *interpreting* clinical performance rather than wrestling with imports.

---

## 1. Load Data and Train Model

Before we can study classification metrics, we need predictions to evaluate. The cell below loads the **breast cancer Wisconsin dataset** (569 samples, 30 features describing cell nuclei — `worst_radius`, `mean_texture`, `mean_concavity`, and others), applies the standard 60/20/20 stratified split, and fits the same Logistic Regression pipeline from the previous notebook.

We then extract both hard predictions (`y_pred_val`) and continuous probabilities (`y_proba_val`) from the validation set. Hard labels let us build a confusion matrix and compute precision/recall; probabilities let us sweep thresholds and plot ROC/PR curves — the full diagnostic toolkit the Health Department needs to assess whether MedScreen is ready for deployment.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset, split it 60/20/20 with stratification, then build and fit a logistic regression pipeline (StandardScaler + LogisticRegression with max_iter=1000). Get hard predictions and class-1 probabilities on the validation set. Print set sizes and validation accuracy."
>
> **After running, verify:**
> - Train/Val/Test sizes are printed
> - Validation accuracy is reported (should be above 95%)
> - Both y_pred_val (hard labels) and y_proba_val (probabilities) are available> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# Split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp)

# Train model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred_val = pipeline.predict(X_val)
y_proba_val = pipeline.predict_proba(X_val)[:, 1]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Validation Accuracy: {pipeline.score(X_val, y_val):.4f}")

**Reading the output:**

The dataset splits into **Train: 341 | Val: 114 | Test: 114**, following the 60/20/20 convention with stratification preserving the ~37% malignant / ~63% benign ratio in every partition. The Logistic Regression pipeline achieves **validation accuracy around 0.97** — impressive on the surface, but the chief medical officer's question is not "What percentage did you get right?" It is "How many malignant tumors did you miss?"

The variables `y_pred_val` (hard 0/1 labels at the default 0.5 threshold) and `y_proba_val` (continuous probabilities for the benign class) are now available. The remainder of this notebook uses these two arrays to compute every metric that answers the board's real questions.

**Key takeaway:** A single accuracy number is only the starting point. The next sections decompose this performance into the four cells of the confusion matrix and the metrics that derive from them.

---

## 2. The Confusion Matrix Deep Dive

Every classification metric the board will see — precision, recall, F1, specificity — is just arithmetic on four numbers. The confusion matrix tabulates those four outcomes for every patient MedScreen evaluated:

```
                    Predicted Negative    Predicted Positive
Actual Negative          TN                     FP
Actual Positive          FN                     TP
```

**In the screening context:**
- **True Positive (TP)**: MedScreen correctly identifies a benign tumor — patient avoids unnecessary intervention
- **True Negative (TN)**: MedScreen correctly flags a malignant tumor — patient receives timely treatment
- **False Positive (FP)**: MedScreen misclassifies a malignant tumor as benign — **missed cancer, potentially catastrophic**
- **False Negative (FN)**: MedScreen flags a benign tumor as malignant — unnecessary biopsy, stressful but survivable

> 💡 **Gemini Prompt:** "Compute the confusion matrix for validation predictions and display it as a heatmap using ConfusionMatrixDisplay with labels Malignant (0) and Benign (1). Unpack TN, FP, FN, TP using cm.ravel() and print each value."
>
> **After running, verify:**
> - Heatmap shows a 2x2 confusion matrix with labeled axes
> - TN, FP, FN, TP are printed as separate named values
> - Most predictions should be on the diagonal (correct classifications)> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_pred_val)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Malignant (0)', 'Benign (1)'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Validation Set')
plt.tight_layout()
plt.show()

TN, FP, FN, TP = cm.ravel()
print("=== CONFUSION MATRIX VALUES ===")
print(f"True Negatives (TN):  {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"True Positives (TP):  {TP}")

**Reading the output:**

The heatmap displays the 2x2 confusion matrix, with rows representing the *actual* diagnosis (pathology-confirmed) and columns representing MedScreen's *predicted* diagnosis. Darker blue squares indicate higher counts — ideally, the diagonal dominates.

Below the plot, the four raw counts are printed:

- **True Negatives (TN):** correctly identified malignant cases — these patients proceed to treatment without delay.
- **False Positives (FP):** malignant cases MedScreen mistakenly called benign — these are the missed cancers. Each one represents a patient who might be sent home without treatment.
- **False Negatives (FN):** benign cases MedScreen mistakenly flagged as malignant — these patients undergo an unnecessary biopsy but ultimately learn they are healthy.
- **True Positives (TP):** correctly identified benign cases — patients confirmed healthy with no further action needed.

Because scikit-learn labels benign as 1 (positive class), the FP cell in its convention corresponds to "predicted benign when actually malignant" — the most dangerous error. In a real clinical deployment, you would typically flip the positive class so that malignant = positive, but here we follow scikit-learn's default.

**Why this matters:** Every metric in the next section is arithmetic on these four counts. The board's question — "How many cancers did we miss?" — maps directly to the FP count (in sklearn's convention). Understanding these four numbers is the foundation for everything that follows.

---

## 3. Core Classification Metrics

Each metric answers a different question the Health Department might ask about MedScreen's performance:

### Precision
**Precision = TP / (TP + FP)**
- "When MedScreen says a tumor is benign, how often is it right?"
- High precision = few patients falsely reassured (few missed cancers among benign predictions)

### Recall (Sensitivity, True Positive Rate)
**Recall = TP / (TP + FN)**
- "Of all truly benign cases, how many did MedScreen correctly identify?"
- High recall = few benign patients unnecessarily alarmed

### F1 Score
**F1 = 2 x (Precision x Recall) / (Precision + Recall)**
- Harmonic mean of precision and recall — useful when the board wants a single number that balances both error types

### Accuracy
**Accuracy = (TP + TN) / (TP + TN + FP + FN)**
- Overall correctness across all patients
- **Warning:** With 63% benign samples, a model that *never detects cancer* scores 63% accuracy

> 💡 **Gemini Prompt:** "Calculate classification metrics manually from TP, TN, FP, FN: Accuracy, Precision, Recall, F1 (using sklearn f1_score), and Specificity. Print all five metrics. Then print sklearn full classification_report with target names Malignant and Benign."
>
> **After running, verify:**
> - Five metrics are printed: Accuracy, Precision, Recall, F1, Specificity
> - classification_report shows per-class precision, recall, f1, and support
> - Manual calculations match sklearn automated values> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Calculate all metrics
metrics = {
    'Accuracy': (TP + TN) / (TP + TN + FP + FN),
    'Precision': TP / (TP + FP) if (TP + FP) > 0 else 0,
    'Recall': TP / (TP + FN) if (TP + FN) > 0 else 0,
    'F1': f1_score(y_val, y_pred_val),
    'Specificity': TN / (TN + FP) if (TN + FP) > 0 else 0
}

print("=== CLASSIFICATION METRICS ===")
for metric, value in metrics.items():
    print(f"{metric:15s}: {value:.4f}")

# Using sklearn functions
print("\n=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_val, y_pred_val, target_names=['Malignant', 'Benign']))

**Reading the output:**

The first block prints five hand-calculated metrics from the confusion matrix:

| Metric | Typical value | What it tells the Health Department |
|--------|--------------|-------------------------------------|
| **Accuracy** | ~0.97 | What fraction of all diagnoses were correct? |
| **Precision** | ~0.97-0.99 | When MedScreen says "benign," how reliable is that? |
| **Recall** | ~0.97-0.99 | Of all truly benign cases, how many did we correctly clear? |
| **F1** | ~0.98 | Single-number summary balancing precision and recall |
| **Specificity** | ~0.93-0.97 | Of all truly malignant cases, how many did we correctly flag? |

The `classification_report` confirms these numbers and adds per-class breakdowns (Malignant row and Benign row) plus macro and weighted averages. The `support` column shows how many validation patients belong to each class.

**Key takeaway:** Precision and recall often tell different stories. For the screening deployment, **specificity** (catching malignant cases) matters most to the oncologist, while **precision for the benign class** matters to patients being told they are healthy. Recall of 0.95 means MedScreen catches 95% of cancers — whether that meets the Health Department's deployment threshold depends on their risk tolerance.

---

## 4. ROC Curve and AUC

The metrics above depend on the 0.5 threshold — change the threshold and every number shifts. The Health Department needs a way to evaluate MedScreen's *overall* discriminative ability, independent of any particular threshold choice. That is exactly what the ROC curve provides.

### ROC (Receiver Operating Characteristic)
- Plots **True Positive Rate** (recall) vs **False Positive Rate** (1 - specificity) as the threshold sweeps from 1.0 to 0.0
- A model that perfectly separates malignant from benign hugs the top-left corner
- A random coin flip follows the diagonal

**AUC (Area Under the Curve) Interpretation:**
- AUC = 1.0: MedScreen perfectly ranks every malignant case above every benign case
- AUC = 0.5: No better than random guessing — the 30 cell features carry no diagnostic signal
- AUC < 0.5: Predictions inverted (would improve by flipping labels)

> 💡 **Gemini Prompt:** "Plot the ROC curve using sklearn roc_curve on validation probabilities. Include the diagonal random-classifier baseline as a dashed line. Compute and display ROC-AUC score in the legend. Label axes as False Positive Rate and True Positive Rate."
>
> **After running, verify:**
> - ROC curve is plotted with AUC value shown in the legend
> - Diagonal dashed line represents a random classifier (AUC=0.5)
> - ROC-AUC score is printed below the plot (should be close to 1.0)> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_val, y_proba_val)
roc_auc = roc_auc_score(y_val, y_proba_val)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")
print("\n💡 Higher AUC = better separation between classes")

**Reading the output:**

The ROC curve plots the **True Positive Rate** (recall for benign) on the y-axis against the **False Positive Rate** (fraction of malignant cases incorrectly called benign) on the x-axis, sweeping the threshold from 1.0 down to 0.0. A perfect screening tool hugs the top-left corner; the dashed diagonal represents a random coin flip that ignores cell morphology entirely.

The **AUC** is printed below the plot. For MedScreen's Logistic Regression on the breast cancer validation set, expect an **ROC-AUC around 0.99**, indicating near-perfect separation between malignant and benign samples. An AUC of 0.99 means: pick one truly benign and one truly malignant patient at random, and MedScreen assigns a higher benign probability to the correct patient 99% of the time.

**Why this matters:** ROC-AUC is a threshold-independent summary of model quality. It is the metric of choice during model *selection* — comparing logistic regression against random forests, for example — because it evaluates the full range of operating points. The Health Department can pick any threshold *after* choosing the best model by AUC.

---

## 5. Precision-Recall Curve

ROC curves can be overly optimistic when classes are imbalanced. Consider a hospital where 95% of aspirates are benign: the False Positive Rate denominator is dominated by the large benign class, masking poor detection of the rare malignant cases. The Precision-Recall (PR) curve focuses exclusively on the positive class, making it a stricter evaluation for screening scenarios.

### When to Use PR Curve vs ROC for MedScreen

**Use ROC when:**
- Classes are roughly balanced (as in our 37/63 dataset)
- You need a threshold-independent model comparison metric

**Use PR Curve when:**
- Classes are heavily imbalanced (e.g., 95% benign in a general-population screening)
- The oncologist cares primarily about the quality of "benign" predictions
- You want to see how precision degrades as you push for higher recall

> 💡 **Gemini Prompt:** "Plot the Precision-Recall curve using sklearn precision_recall_curve. Add a horizontal baseline at the positive class ratio. Compute average_precision_score and show it in the legend. Label axes as Recall (x) and Precision (y)."
>
> **After running, verify:**
> - PR curve is plotted with Average Precision (AP) shown in the legend
> - Horizontal dashed baseline shows the positive class ratio
> - AP score is printed below the plot> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Precision-Recall Curve
precision, recall, thresholds_pr = precision_recall_curve(y_val, y_proba_val)
pr_auc = average_precision_score(y_val, y_proba_val)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, linewidth=2, label=f'PR Curve (AP = {pr_auc:.3f})')
baseline = y_val.sum() / len(y_val)
plt.axhline(y=baseline, color='k', linestyle='--', linewidth=1, label=f'Baseline (class ratio = {baseline:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average Precision (PR-AUC): {pr_auc:.4f}")
print("\n💡 Closer to top-right = better model")

**Reading the output:**

The Precision-Recall curve plots **Precision** on the y-axis against **Recall** on the x-axis. The dashed horizontal baseline represents the positive-class prevalence (roughly 0.63 for benign) — a model that stamps every aspirate "benign" would land on that line. A useful screening tool must stay well above it.

The **Average Precision (AP)** score, also called PR-AUC, summarizes the area under this curve. For MedScreen, expect **AP around 0.99**, mirroring the strong ROC-AUC. The curve hugs the top-right corner, meaning the model maintains high precision even at high recall — it rarely mislabels a malignant case as benign, even when catching nearly every benign case correctly.

**Key takeaway:** PR curves become the primary evaluation tool when MedScreen scales to general-population screening, where 95%+ of aspirates are benign. In that setting, ROC-AUC can look deceptively good because the massive number of true negatives inflates the denominator of the False Positive Rate. PR-AUC focuses on the question that matters: "Of the patients we told are healthy, how many actually have cancer?"

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Build a threshold sweep and pick a threshold by business cost.

**Scenario:** Medical diagnosis
- False Negative (missed cancer): Cost = \$50,000 (late treatment)
- False Positive (false alarm): Cost = \$1,000 (unnecessary biopsy)

---

> 💡 **Gemini Prompt:** "Sweep 50 thresholds from 0.1 to 0.9. For each, compute TP, FP, FN, TN, Precision, Recall, and total cost using cost_FN=50000 and cost_FP=1000. Find the threshold that minimizes total cost. Plot two side-by-side charts: (1) Total Cost vs Threshold with the optimal threshold marked, and (2) Precision and Recall vs Threshold."
>
> **After running, verify:**
> - Optimal threshold and its expected cost are printed
> - Left plot shows a cost curve with the minimum marked by a red dashed line
> - Right plot shows precision increasing and recall decreasing as threshold rises> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance


### YOUR ANALYSIS:

**Question 1: Why is the optimal threshold different from 0.5?**  
[Your answer - think about asymmetric costs]

**Question 2: What happens if you change the cost ratio?**  
[Your answer - try FN cost = \$100,000]

**Question 3: In production, how would you monitor this?**  
[Your answer - what could go wrong over time?]

---

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Explain why accuracy fails under imbalance (with evidence).

---

> 💡 **Gemini Prompt:** "Create a synthetic imbalanced dataset with make_classification: 1000 samples, 20 features, 95%/5% class split. Split 70/30 with stratification, fit a logistic regression pipeline, and print the classification_report. Compare against a naive all-zeros baseline to show that high accuracy can be misleading with class imbalance."
>
> **After running, verify:**
> - Class distribution confirms ~95% class 0 and ~5% class 1
> - classification_report shows per-class metrics (class 1 recall may be low)
> - Naive baseline accuracy is ~95% despite predicting all zeros (zero recall for class 1)> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance


### YOUR EXPLANATION:

**Why accuracy is misleading:**  
[Your explanation with evidence from above]

**Better metrics for imbalance:**  
[Which metrics would you use instead?]

**Real-world example:**  
[Give an example where this matters]

---

## 6. Metrics Dashboard

Individual metrics tell part of the story; a **dashboard** tells the whole story at a glance. When the Health Department board reviews MedScreen's readiness for deployment, they need accuracy, precision, recall, F1, specificity, ROC-AUC, and PR-AUC side by side — not scattered across separate code cells.

The function below wraps every metric we have discussed into a single call that returns a dictionary. You can log it, compare it across models, or embed it directly in a board presentation. Building this reusable dashboard now guarantees that every model you evaluate — logistic regression, random forest, gradient boosting — is measured on the same terms, making apples-to-apples comparison straightforward.

> 💡 **Gemini Prompt:** "Define a function create_metrics_dashboard(y_true, y_pred, y_proba) that computes Accuracy, Precision, Recall, F1, Specificity, ROC-AUC, PR-AUC, and the four confusion matrix values (TP, FP, FN, TN), returning them as a dictionary. Call it on the validation set and print all metrics in a formatted dashboard."
>
> **After running, verify:**
> - Function returns a dictionary with 11 keys (7 metrics + 4 confusion matrix counts)
> - All performance metrics are printed in a clean formatted layout
> - ROC-AUC and PR-AUC are included alongside threshold-dependent metrics> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Complete metrics dashboard
def create_metrics_dashboard(y_true, y_pred, y_proba):
    """Generate comprehensive classification metrics"""
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    
    metrics = {
        'Accuracy': (TP + TN) / (TP + TN + FP + FN),
        'Precision': TP / (TP + FP) if (TP + FP) > 0 else 0,
        'Recall': TP / (TP + FN) if (TP + FN) > 0 else 0,
        'F1': f1_score(y_true, y_pred),
        'Specificity': TN / (TN + FP) if (TN + FP) > 0 else 0,
        'ROC_AUC': roc_auc_score(y_true, y_proba),
        'PR_AUC': average_precision_score(y_true, y_proba),
        'TP': TP,
        'FP': FP,
        'FN': FN,
        'TN': TN
    }
    return metrics

dashboard = create_metrics_dashboard(y_val, y_pred_val, y_proba_val)

print("=== COMPREHENSIVE METRICS DASHBOARD ===")
print(f"\nPerformance Metrics:")
print(f"  Accuracy:    {dashboard['Accuracy']:.4f}")
print(f"  Precision:   {dashboard['Precision']:.4f}")
print(f"  Recall:      {dashboard['Recall']:.4f}")
print(f"  F1 Score:    {dashboard['F1']:.4f}")
print(f"  Specificity: {dashboard['Specificity']:.4f}")
print(f"  ROC-AUC:     {dashboard['ROC_AUC']:.4f}")
print(f"  PR-AUC:      {dashboard['PR_AUC']:.4f}")

print(f"\nConfusion Matrix:")
print(f"  TP: {dashboard['TP']:4d}    FP: {dashboard['FP']:4d}")
print(f"  FN: {dashboard['FN']:4d}    TN: {dashboard['TN']:4d}")

**Reading the output:**

The dashboard prints two sections from a single function call:

1. **Performance Metrics:** Accuracy, Precision, Recall, F1, Specificity, ROC-AUC, and PR-AUC — all computed on the breast cancer validation set. Expect values in the **0.95-0.99** range for this well-separated dataset. Together, these seven numbers give the board a complete picture: overall correctness (accuracy), reliability of benign predictions (precision), cancer-detection rate (specificity), and threshold-independent model quality (ROC-AUC, PR-AUC).
2. **Confusion Matrix counts:** TP, FP, FN, TN as raw integers so any stakeholder can verify a metric by hand or drill into the specific patients behind each error.

This `create_metrics_dashboard` function is designed to be *reusable*: call it for any model and get a standardized report. In your course project, you will invoke it for each candidate model and collect the results into a comparison DataFrame — the same table the Health Department would present to the review board before approving MedScreen for statewide deployment.

**Why this matters:** A metrics dashboard eliminates the risk of forgetting a metric or computing it inconsistently across models. It is the final step before stakeholder presentation, ensuring every candidate is evaluated on exactly the same terms.

---

## 7. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Confusion Matrix**: Foundation for understanding classification errors
2. **Precision vs Recall**: Tradeoff between false positives and false negatives
3. **ROC and PR Curves**: Visualize performance across thresholds
4. **Cost-Based Thresholding**: Align decisions to business objectives
5. **Imbalance Handling**: Accuracy is dangerous — use precision, recall, AUC

### Critical Rules:

> **"Never trust accuracy alone"**

> **"Choose thresholds based on business costs, not defaults"**

> **"With imbalance, use PR curves over ROC curves"**

### Next Steps:

- **Next notebook (NB08):** Cross-validation for robust model comparison — today's metrics plug directly into the `scoring=...` parameter of scikit-learn's CV routines.
- **Later in the course (NB16):** Calibration — whether the probabilities we are tuning are trustworthy — comes back once we meet classifiers (random forests, gradient boosting) that can actually be miscalibrated.
- Apply today's metrics dashboard to your project dataset.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- Fawcett, T. (2006). "An introduction to ROC analysis." *Pattern Recognition Letters*, 27(8), 861-874.
- Saito, T., & Rehmsmeier, M. (2015). "The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets." *PLOS ONE*.
- scikit-learn User Guide: [Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* - Chapter on evaluation and costs

---



<center>

Thank you!

</center>